# Data Description
- Author: RUEI-HSIANG, CHANG（B12303030；AI協助）
- Date first created: 20261019
- Background: 批次交叉核對全部692筆議員名錄線索，整理歷史身分證據與申報學歷；舊盤點檔案不覆寫。
- Unit of observation: review=一候選人；history=一候選人×2022紀錄；exceptions=一候選人×待查項目；master=一登記者；fields=一登記者×欄位。
- Sample period: 2026登記、內政部2026-10-19擷取的111年選舉人物頁、2022歷史選舉紀錄。
- Known issues: 同名同縣市、同編號選區或正式公告跨屆對照、當選紀錄一致僅提供身分連結支持，不是獨立身分證明；不同屆相同編號不保證地理邊界相同。新增補選遞補佐證包含完整決議、公告名單、2筆官方標題及1筆議會刊載轉載新聞，證據層級分列；未以當選推定2026現任，也未確認2026資格。學歷是來源申報而非學位認證。未補值、刪人、改研究樣本。現任與首次參選尚須另外核對。
- Definition for each variable:
  - review保留原來源線索欄位：candidate_key/name/county、directory_key、profile_url、education_raw/experience_raw、source_file/sha256與擷取狀態；match_status更新為本輪結果。
  - electoral_district_2026為登記選區。identity_review_result為查核分類；identity_link_supported表示三來源紀錄是否依明訂規則一致；supplement_event/source_url/source_file/source_locator/verification為補選遞補佐證及證據層級；history_match_count為同縣市姓名或有明文別名證據的2022候選紀錄數；matched_alias/alias_evidence_url僅記官方更名證據。
  - history_*為2022原申報：姓名、縣市、選區編號、民國生日、性別、學歷、當選符號及當時現任標記；history_file/row/archive_member是精確原文定位。*與!皆當選，!表示婦女保障；空白或-不自動推定日後未遞補。
  - history_district_same_number只比較編號，不宣稱跨屆邊界不變；district_link_supported含正式公告對照，district_match_method區分同編號與公告對照，district_crosswalk_source為公告來源。education_group_2022_declared僅在身分連結有支持時依2022分類轉為大學以下／大學／研究所，非當屆學歷認證。
  - moi_explicit_degree_candidate取原文明確學士／碩士／博士字樣中的最高層級，副學士獨立於學士，排除含在學、肄業、候選、研究班、學程班、在職專班、論文口試、學分、榮譽的子句；moi_excluded_non_degree_clauses列排除原文。未明說學位的不推定畢業。
  - education_comparison比較來源層級：一致、年代或層級差異、原文待判讀、缺失。degree_authentication明示尚未認證；incumbency_2026全空白，incumbency_review說明本輪界線；review_date為本輪日期。
  - history中same_district_number與source_url提供歷史定位。exceptions的field/reason/next_action表示需補查的項目；不是刪除清單。
  - master/fields沿用原盤點資料字典，僅更新692位的profile與education狀態及證據；其他狀態不變。換行可逆轉義為文字\n，一筆CSV資料占一個實體行。
- Source data: 詳見處理Notebook固定INPUTS（六份上游／manifest、六份2022候選CSV及選區與補選遞補補充來源），含逐檔SHA-256。
- Processing code: `data/code/verify_council_identity_education_20261019.ipynb`
- Role: identity-education-review / collection-inventory；非分析訓練樣本。
- Verification status: 下方clean-kernel實際檢查schema、筆數、鍵、hash與邏輯；生產端已重讀CSV逐值比對。
- Data path / SHA-256 (review): `data/processed/council_identity_education_review_20261019.csv` / `becae07e111c6f99203902c0759dd12c7b7f9d21e029e961e24c12bf6467524e`；692列。
- Data path / SHA-256 (history): `data/processed/council_history_matches_20261019.csv` / `2215ca851d344d377e5cf66f4a66e83d062baeb25c4d278a8279d0b93dff6eb0`；690列。
- Data path / SHA-256 (exceptions): `data/processed/council_review_exceptions_20261019.csv` / `44395328a1bf27968aafd17679a5f5a458822e59eeedeadf78dc85654b150414`；434列。
- Data path / SHA-256 (master): `data/processed/candidate_master_reviewed_20261019.csv` / `8beb9e6d8c766f9b1484f5c2bc71a1bbf3702640be2135a3abb907d1e6e96177`；1583列。
- Data path / SHA-256 (fields): `data/processed/candidate_fields_reviewed_20261019.csv` / `ff10c75856103a246126dd38ce4b274c647265dc7f4c1241cfcd6f5daae3e41c`；31660列。
- Downstream analysis input: 尚無；只用於來源核對與資料準備。


In [1]:
from pathlib import Path
import hashlib,json
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').is_file() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()


## Change for Your Own Data
固定輸出路徑與hash；空值保留、全欄字串讀取。

In [2]:
SPECS={'review': {'path': 'data/processed/council_identity_education_review_20261019.csv', 'rows': 692, 'columns': ['candidate_key', 'candidate_name', 'county_city', 'directory_key', 'profile_url', 'match_status', 'education_raw', 'experience_raw', 'http_status', 'page_contains_name', 'source_file', 'sha256', 'capture_error', 'captured_date', 'electoral_district_2026', 'identity_review_result', 'identity_link_supported', 'supplement_event', 'supplement_source_url', 'supplement_source_file', 'supplement_source_locator', 'supplement_verification', 'history_match_count', 'matched_alias', 'alias_evidence_url', 'history_district_same_number', 'district_link_supported', 'district_match_method', 'district_crosswalk_source', 'history_education_2022', 'education_group_2022_declared', 'moi_explicit_degree_candidate', 'moi_excluded_non_degree_clauses', 'education_comparison', 'degree_authentication', 'incumbency_2026', 'incumbency_review', 'review_date', 'history_name', 'history_county', 'history_district', 'history_birth_roc', 'history_sex', 'history_education', 'history_elected_marker', 'history_incumbency_at_2022', 'history_file', 'history_row', 'history_archive_member'], 'sha256': 'becae07e111c6f99203902c0759dd12c7b7f9d21e029e961e24c12bf6467524e'}, 'history': {'path': 'data/processed/council_history_matches_20261019.csv', 'rows': 690, 'columns': ['candidate_key', 'candidate_name', 'electoral_district_2026', 'history_name', 'history_county', 'history_district', 'history_birth_roc', 'history_sex', 'history_education', 'history_elected_marker', 'history_incumbency_at_2022', 'history_file', 'history_row', 'history_archive_member', 'same_district_number', 'source_url'], 'sha256': '2215ca851d344d377e5cf66f4a66e83d062baeb25c4d278a8279d0b93dff6eb0'}, 'exceptions': {'path': 'data/processed/council_review_exceptions_20261019.csv', 'rows': 434, 'columns': ['candidate_key', 'candidate_name', 'county_city', 'field', 'reason', 'profile_url', 'education_raw', 'history_education_2022', 'next_action'], 'sha256': '44395328a1bf27968aafd17679a5f5a458822e59eeedeadf78dc85654b150414'}, 'master': {'path': 'data/processed/candidate_master_reviewed_20261019.csv', 'rows': 1583, 'columns': ['candidate_key', 'candidate_name', 'office_type', 'county_city', 'electoral_district', 'party_registration_raw', 'registration_source', 'registration_source_file', 'registration_source_page', 'qualification_inherited', 'qualification_audit', 'sample_inclusion', 'directory_lead_count', 'updated_date', 'sex_status', 'birth_date_status', 'education_status', 'incumbency_status', 'first_time_this_office_status', 'first_time_any_office_status', 'profile_status', 'website_status', 'Facebook_status', 'Instagram_status', 'Threads_status', 'posts_Facebook_status', 'posts_Instagram_status', 'posts_Threads_status', 'followers_status', 'policy_text_status', 'visual_status', 'political_family_status', 'administrative_experience_status', 'qualification_status'], 'sha256': '8beb9e6d8c766f9b1484f5c2bc71a1bbf3702640be2135a3abb907d1e6e96177'}, 'fields': {'path': 'data/processed/candidate_fields_reviewed_20261019.csv', 'rows': 31660, 'columns': ['candidate_key', 'candidate_name', 'office_type', 'county_city', 'field', 'status', 'existing_value', 'evidence', 'note', 'next_source', 'updated_date'], 'sha256': 'ff10c75856103a246126dd38ce4b274c647265dc7f4c1241cfcd6f5daae3e41c'}}
data={}
for key,s in SPECS.items():
 p=ROOT/s['path'];assert sha(p)==s['sha256']
 data[key]=pd.read_csv(p,dtype=str,keep_default_na=False)
 assert list(data[key].columns)==s['columns'] and len(data[key])==s['rows']


## Summary

In [3]:
review=data['review']
display(review.identity_review_result.value_counts().to_frame('筆數'))
display(review.education_comparison.value_counts().to_frame('筆數'))
print('歷史學歷分組可用',sum(review.education_group_2022_declared!=''),'身分仍待補查',sum(review.identity_link_supported!='True'))

,筆數
identity_review_result,
三來源紀錄一致，身分連結有支持,653
官方選區變更對照後紀錄一致,25
補選／遞補補充來源支持身分連結,11
官方公報生日學歷佐證字形差異，紀錄一致,1
官方更名證據及三來源紀錄一致,1
未找到2022同名紀錄，待補查,1


,筆數
education_comparison,
官方原文已有，學位層級仍待判讀,317
兩來源申報學歷層級一致,295
內政部學歷空白，歷史申報另列,77
來源年代或層級差異，待複核,3


歷史學歷分組可用 689 身分仍待補查 1


### Sample

In [4]:
for key,df in data.items():
 print(key);display(df.head(10))

review


,candidate_key,candidate_name,county_city,directory_key,profile_url,match_status,education_raw,experience_raw,http_status,page_contains_name,...,history_county,history_district,history_birth_roc,history_sex,history_education,history_elected_marker,history_incumbency_at_2022,history_file,history_row,history_archive_member
0,2026|直轄市議員|臺北市第1選舉區|鍾佩玲,鍾佩玲,臺北市,ER11112AA00008,https://www.moi.gov.tw/LocalOfficial_Content.a...,三來源紀錄一致，身分連結有支持,國立政治大學經營管理碩士\n世新大學口語傳播學系,壹電視新聞主播\n三立新聞主播\n臺灣新動力智庫教育文化組召集人\n臺灣綠色文化協會執行長\...,200,True,...,臺北市,1,0680327,女,碩士,*,Y,data/raw/cec_all_offices/107_elcand.csv,10,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...
1,2026|直轄市議員|臺北市第1選舉區|陳賢蔚,陳賢蔚,臺北市,ER11112AA00002,https://www.moi.gov.tw/LocalOfficial_Content.a...,三來源紀錄一致，身分連結有支持,國立臺灣海洋大學工學學士\n臺北市立復興高級中學,國立臺灣大學政治學系碩士生\n亞太自由婦女協會理事\n亞洲政經與和平交流協會理事\n中華民國...,200,True,...,臺北市,1,0660712,男,大學,*,Y,data/raw/cec_all_offices/107_elcand.csv,2,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...
2,2026|直轄市議員|臺北市第1選舉區|林世宗,林世宗,臺北市,ER11112AA00005,https://www.moi.gov.tw/LocalOfficial_Content.a...,三來源紀錄一致，身分連結有支持,,,200,True,...,臺北市,1,0480613,男,碩士,*,Y,data/raw/cec_all_offices/107_elcand.csv,6,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...
3,2026|直轄市議員|臺北市第1選舉區|黃瀞瑩,黃瀞瑩,臺北市,ER11112AA00010,https://www.moi.gov.tw/LocalOfficial_Content.a...,三來源紀錄一致，身分連結有支持,倫敦大學金匠學院政治傳播碩士\n天主教輔仁大學新聞傳播學系學士\n臺北市立明倫高中\n蘭雅國...,臺北市政府副發言人\n黃珊珊副市長辦公室秘書\n臺北市政府媒體事務組\nETtoday 要聞組記者,200,True,...,臺北市,1,0810610,女,碩士,*,N,data/raw/cec_all_offices/107_elcand.csv,13,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...
4,2026|直轄市議員|臺北市第1選舉區|陳慈慧,陳慈慧,臺北市,ER11112AA00007,https://www.moi.gov.tw/LocalOfficial_Content.a...,三來源紀錄一致，身分連結有支持,開南大學應用日語學系\n泰北高中\n士林區福安國中\n士林區社子國小,陽明山國際青商會副會長\n台灣棒球聯盟理事長,200,True,...,臺北市,1,0721122,女,大學,*,Y,data/raw/cec_all_offices/107_elcand.csv,9,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...
5,2026|直轄市議員|臺北市第1選舉區|陳重文,陳重文,臺北市,ER11112AA00006,https://www.moi.gov.tw/LocalOfficial_Content.a...,三來源紀錄一致，身分連結有支持,國立體育大學體研所博士\n國立體育大學體育推廣學系碩士,臺北市立大學競技研究所、陸上系技擊系助理教授\n臺北市健美健身運動協會 理事長\n臺北市沙灘...,200,True,...,臺北市,1,0660924,男,博士,*,Y,data/raw/cec_all_offices/107_elcand.csv,7,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...
6,2026|直轄市議員|臺北市第1選舉區|林杏兒,林杏兒,臺北市,ER11112AA00003,https://www.moi.gov.tw/LocalOfficial_Content.a...,三來源紀錄一致，身分連結有支持,銘傳大學公共事務學系碩士\n福安國中\n社子國小,中國國民黨副發言人\n台北市七星獅子會會長\n北投民防大隊副大隊長,200,True,...,臺北市,1,0700105,女,碩士,*,N,data/raw/cec_all_offices/107_elcand.csv,4,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...
7,2026|直轄市議員|臺北市第1選舉區|張斯綱,張斯綱,臺北市,ER11112AA00011,https://www.moi.gov.tw/LocalOfficial_Content.a...,三來源紀錄一致，身分連結有支持,國立政治大學東亞所碩士\n輔仁大學法律學系財經組,中國國民黨發言人 \n中國國民黨文傳會副主委\n盛弘醫藥股份有限公司副總經理兼發言人\n衛生...,200,True,...,臺北市,1,0600909,男,碩士,*,Y,data/raw/cec_all_offices/107_elcand.csv,20,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...
8,2026|直轄市議員|臺北市第1選舉區|汪志冰,汪志冰,臺北市,ER11112AA00004,https://www.moi.gov.tw/LocalOfficial_Content.a...,三來源紀錄一致，身分連結有支持,美國東密西根大學文學碩士\n東吳大學德文系\n文藻外語學院德文科,中華民國全國總工會秘書長\n經濟部航太工業推動小組經理\n經濟部投資業務處專員\n日本亞太經...,200,True,...,臺北市,1,0460420,女,碩士,*,Y,data/raw/cec_all_offices/107_elcand.csv,5,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...
9,2026|直轄市議員|臺北市第1選舉區|林延鳳,林延鳳,臺北市,ER11112AA00001,https://www.moi.gov.tw/LocalOfficial_Content.a...,三來源紀錄一致，身分連結有支持,臺灣大學生物產業傳播暨發展學系碩士,八大電視台行銷公關\n民進黨婦女部專員\n台北市友善空間協會秘書長,200,True,...,臺北市,1,0710417,女,碩士,*,N,data/raw/cec_all_offices/107_elcand.csv,1,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...


history


,candidate_key,candidate_name,electoral_district_2026,history_name,history_county,history_district,history_birth_roc,history_sex,history_education,history_elected_marker,history_incumbency_at_2022,history_file,history_row,history_archive_member,same_district_number,source_url
0,2026|直轄市議員|臺北市第1選舉區|鍾佩玲,鍾佩玲,臺北市第1選舉區,鍾佩玲,臺北市,1,0680327,女,碩士,*,Y,data/raw/cec_all_offices/107_elcand.csv,10,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,True,https://db.cec.gov.tw/
1,2026|直轄市議員|臺北市第1選舉區|陳賢蔚,陳賢蔚,臺北市第1選舉區,陳賢蔚,臺北市,1,0660712,男,大學,*,Y,data/raw/cec_all_offices/107_elcand.csv,2,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,True,https://db.cec.gov.tw/
2,2026|直轄市議員|臺北市第1選舉區|林世宗,林世宗,臺北市第1選舉區,林世宗,臺北市,1,0480613,男,碩士,*,Y,data/raw/cec_all_offices/107_elcand.csv,6,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,True,https://db.cec.gov.tw/
3,2026|直轄市議員|臺北市第1選舉區|黃瀞瑩,黃瀞瑩,臺北市第1選舉區,黃瀞瑩,臺北市,1,0810610,女,碩士,*,N,data/raw/cec_all_offices/107_elcand.csv,13,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,True,https://db.cec.gov.tw/
4,2026|直轄市議員|臺北市第1選舉區|陳慈慧,陳慈慧,臺北市第1選舉區,陳慈慧,臺北市,1,0721122,女,大學,*,Y,data/raw/cec_all_offices/107_elcand.csv,9,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,True,https://db.cec.gov.tw/
5,2026|直轄市議員|臺北市第1選舉區|陳重文,陳重文,臺北市第1選舉區,陳重文,臺北市,1,0660924,男,博士,*,Y,data/raw/cec_all_offices/107_elcand.csv,7,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,True,https://db.cec.gov.tw/
6,2026|直轄市議員|臺北市第1選舉區|林杏兒,林杏兒,臺北市第1選舉區,林杏兒,臺北市,1,0700105,女,碩士,*,N,data/raw/cec_all_offices/107_elcand.csv,4,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,True,https://db.cec.gov.tw/
7,2026|直轄市議員|臺北市第1選舉區|張斯綱,張斯綱,臺北市第1選舉區,張斯綱,臺北市,1,0600909,男,碩士,*,Y,data/raw/cec_all_offices/107_elcand.csv,20,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,True,https://db.cec.gov.tw/
8,2026|直轄市議員|臺北市第1選舉區|汪志冰,汪志冰,臺北市第1選舉區,汪志冰,臺北市,1,0460420,女,碩士,*,Y,data/raw/cec_all_offices/107_elcand.csv,5,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,True,https://db.cec.gov.tw/
9,2026|直轄市議員|臺北市第1選舉區|林延鳳,林延鳳,臺北市第1選舉區,林延鳳,臺北市,1,0710417,女,碩士,*,N,data/raw/cec_all_offices/107_elcand.csv,1,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,True,https://db.cec.gov.tw/


exceptions


,candidate_key,candidate_name,county_city,field,reason,profile_url,education_raw,history_education_2022,next_action
0,2026|直轄市議員|臺北市第1選舉區|林世宗,林世宗,臺北市,education,內政部學歷空白，歷史申報另列,https://www.moi.gov.tw/LocalOfficial_Content.a...,,碩士,身分異動另查官方履歷、選區公告；學歷回查畢業與學位原文，未查清保留空白
1,2026|直轄市議員|臺北市第1選舉區|陳慈慧,陳慈慧,臺北市,education,官方原文已有，學位層級仍待判讀,https://www.moi.gov.tw/LocalOfficial_Content.a...,開南大學應用日語學系\n泰北高中\n士林區福安國中\n士林區社子國小,大學,身分異動另查官方履歷、選區公告；學歷回查畢業與學位原文，未查清保留空白
2,2026|直轄市議員|臺北市第2選舉區|游淑慧,游淑慧,臺北市,education,官方原文已有，學位層級仍待判讀,https://www.moi.gov.tw/LocalOfficial_Content.a...,臺灣大學政治系\n中山女高,大學,身分異動另查官方履歷、選區公告；學歷回查畢業與學位原文，未查清保留空白
3,2026|直轄市議員|臺北市第2選舉區|陳宥丞,陳宥丞,臺北市,education,官方原文已有，學位層級仍待判讀,https://www.moi.gov.tw/LocalOfficial_Content.a...,麗山國小\n西湖國中\n景文高中\n真理大學財經法律學系,大學,身分異動另查官方履歷、選區公告；學歷回查畢業與學位原文，未查清保留空白
4,2026|直轄市議員|臺北市第3選舉區|張文潔,張文潔,臺北市,education,官方原文已有，學位層級仍待判讀,https://www.moi.gov.tw/LocalOfficial_Content.a...,國立臺北大學企業管理學系\n台北市私立東山高中\n台北市立西松國中\n台北市立西松國小,大學,身分異動另查官方履歷、選區公告；學歷回查畢業與學位原文，未查清保留空白
5,2026|直轄市議員|臺北市第3選舉區|洪健益,洪健益,臺北市,education,官方原文已有，學位層級仍待判讀,https://www.moi.gov.tw/LocalOfficial_Content.a...,永春國小\n永吉國中\n祐德高中\n中華技術學院二技部\n崇右技術學院二專部\n淡江大學大陸研究所,碩士,身分異動另查官方履歷、選區公告；學歷回查畢業與學位原文，未查清保留空白
6,2026|直轄市議員|臺北市第4選舉區|顏若芳,顏若芳,臺北市,education,官方原文已有，學位層級仍待判讀,https://www.moi.gov.tw/LocalOfficial_Content.a...,文化大學政治系\n百齡高中\n建成國中\n靜心國小,大學,身分異動另查官方履歷、選區公告；學歷回查畢業與學位原文，未查清保留空白
7,2026|直轄市議員|臺北市第4選舉區|陳炳甫,陳炳甫,臺北市,education,官方原文已有，學位層級仍待判讀,https://www.moi.gov.tw/LocalOfficial_Content.a...,臺灣大學商學研究所\n日本桐蔭學園橫濱大學\n臺北市私立泰北高中\n臺北市市立介壽國中\n臺...,碩士,身分異動另查官方履歷、選區公告；學歷回查畢業與學位原文，未查清保留空白
8,2026|直轄市議員|臺北市第4選舉區|柳采葳,柳采葳,臺北市,education,官方原文已有，學位層級仍待判讀,https://www.moi.gov.tw/LocalOfficial_Content.a...,銘傳大學新聞學系,大學,身分異動另查官方履歷、選區公告；學歷回查畢業與學位原文，未查清保留空白
9,2026|直轄市議員|臺北市第4選舉區|林亮君,林亮君,臺北市,education,官方原文已有，學位層級仍待判讀,https://www.moi.gov.tw/LocalOfficial_Content.a...,中山女高\n國立臺灣師範大學企管系 \n國立清華大學科技法律所,碩士,身分異動另查官方履歷、選區公告；學歷回查畢業與學位原文，未查清保留空白


master


,candidate_key,candidate_name,office_type,county_city,electoral_district,party_registration_raw,registration_source,registration_source_file,registration_source_page,qualification_inherited,...,Threads_status,posts_Facebook_status,posts_Instagram_status,posts_Threads_status,followers_status,policy_text_status,visual_status,political_family_status,administrative_experience_status,qualification_status
0,2026-C1-001,郭璽,直轄市長,臺北市,,台灣麻將最大黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
1,2026-C1-002,沈伯洋,直轄市長,臺北市,,民主進步黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,部分原文已保存，尚未政策編碼,未查找,未查找,未查找,官方內文待重新核對
2,2026-C1-003,蕭文乾,直轄市長,臺北市,,臺灣SoR無法黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
3,2026-C1-004,唐新民,直轄市長,臺北市,,三勢團結促進聯盟,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
4,2026-C1-005,蔣萬安,直轄市長,臺北市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
5,2026-C1-006,林志成,直轄市長,臺北市,,無,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
6,2026-C1-007,蘇巧慧,直轄市長,新北市,,民主進步黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,部分原文已保存，尚未政策編碼,未查找,未查找,未查找,官方內文待重新核對
7,2026-C1-008,李四川,直轄市長,新北市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
8,2026-C1-009,蘇輝湟,直轄市長,新北市,,無,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
9,2026-C1-010,張善政,直轄市長,桃園市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對


fields


,candidate_key,candidate_name,office_type,county_city,field,status,existing_value,evidence,note,next_source,updated_date
0,2026-C1-001,郭璽,直轄市長,臺北市,sex,已有資料，待來源複核,男,https://db.cec.gov.tw/,dob_corroborated_by_media,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
1,2026-C1-001,郭璽,直轄市長,臺北市,birth_date,已有資料，待來源複核,1958-08-31,https://db.cec.gov.tw/,CEC_name_and_birth_date_corroborated_by_CNA,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
2,2026-C1-001,郭璽,直轄市長,臺北市,education,已有資料，待來源複核,大學,,歷史公報或媒體補充，非2026學歷認證,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
3,2026-C1-001,郭璽,直轄市長,臺北市,incumbency,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
4,2026-C1-001,郭璽,直轄市長,臺北市,first_time_this_office,既有搜尋後仍缺，待補查,,,0=有過參選證據；空白不表示首次。此處沿用既有表。,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
5,2026-C1-001,郭璽,直轄市長,臺北市,first_time_any_office,已有資料，待來源複核,0,https://db.cec.gov.tw/,0=有過參選證據；空白不表示首次。此處沿用既有表。,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
6,2026-C1-001,郭璽,直轄市長,臺北市,profile,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
7,2026-C1-001,郭璽,直轄市長,臺北市,website,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
8,2026-C1-001,郭璽,直轄市長,臺北市,Facebook,尚未驗證，不代表無帳號,,,,官方人物頁／競選網站交叉連結,2026-10-19
9,2026-C1-001,郭璽,直轄市長,臺北市,Instagram,尚未驗證，不代表無帳號,,,,官方人物頁／競選網站交叉連結,2026-10-19


### Summary Stats

In [5]:
display(pd.crosstab(review.county_city,review.identity_review_result))
display(review.education_group_2022_declared.value_counts(dropna=False).to_frame('筆數'))
display(data['exceptions'].groupby(['field','reason']).size().rename('N').to_frame())
print('分母為692位，exceptions一人可多列；空白非無學歷。')

identity_review_result,三來源紀錄一致，身分連結有支持,官方公報生日學歷佐證字形差異，紀錄一致,官方更名證據及三來源紀錄一致,官方選區變更對照後紀錄一致,未找到2022同名紀錄，待補查,補選／遞補補充來源支持身分連結
county_city,,,,,,
南投縣,23,0,0,0,0,1
嘉義市,17,0,0,0,1,0
嘉義縣,29,0,0,0,0,0
基隆市,23,0,0,0,0,0
宜蘭縣,23,0,0,0,0,0
屏東縣,37,0,0,0,0,1
彰化縣,37,0,1,0,0,2
新北市,48,0,0,0,0,0
新竹市,27,0,0,0,0,0


,筆數
education_group_2022_declared,
研究所,343
大學,190
大學以下,156
,3


N
field                reason                 
education            來源年代或層級差異，待複核         3
                     內政部學歷空白，歷史申報另列       77
                     官方原文已有，學位層級仍待判讀     317
education_non_degree 含在學／肄業／學分等字樣，未升格學位   36
identity             未找到2022同名紀錄，待補查       1

分母為692位，exceptions一人可多列；空白非無學歷。


### Assertions

In [6]:
assert len(review)==692 and review.candidate_key.is_unique
assert len(data['master'])==1583 and data['master'].candidate_key.is_unique
assert not data['fields'].duplicated(['candidate_key','field']).any()
assert set(review.candidate_key)<=set(data['master'].candidate_key)
assert (review.loc[review.identity_link_supported=='True','history_elected_marker'].isin(['*','!']) | (review.loc[review.identity_link_supported=='True','supplement_event']!='')).all()
assert (review.incumbency_2026=='').all()
for key,s in SPECS.items():
 assert sha(ROOT/s['path'])==s['sha256']
 assert len((ROOT/s['path']).read_text(encoding='utf-8-sig').splitlines())==s['rows']+1
print('通過schema／筆數／鍵／當選符號／單行CSV與hash；非學位、在任及2026資格認證。')

通過schema／筆數／鍵／當選符號／單行CSV與hash；非學位、在任及2026資格認證。
